<a href="https://colab.research.google.com/github/GimenesPaula/GimenesPaula/blob/main/lanc_vendas_anos_Focus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bibliotecas Phyton

In [1]:
!pip install requests

In [2]:
pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.7 MB/s eta 0:00:00


In [3]:
#Carrega Bibliotecas
import pandas as pd
import numpy as np
import re
from functools import lru_cache
from unidecode import unidecode
import requests

# Fazer Upload planilhas
1.   Lançamentos
2.   Inci
3.   Vendas

In [4]:
#Lançamentos
from google.colab import files
uploaded = files.upload()

Saving MIntel 2025 Br, Ar, Ch.xlsx to MIntel 2025 Br, Ar, Ch.xlsx


In [5]:
filename = next(iter(uploaded))

In [6]:
# INCI name produtos
from google.colab import files
inci = files.upload()

Saving inci_16_08.xlsx to inci_16_08.xlsx


In [7]:
filename2 = next(iter(inci))

In [690]:
#Vendas Distribuidores
from google.colab import files
dist = files.upload()

Saving Focus Sales Report Q4 2025.xlsx to Focus Sales Report Q4 2025 (1).xlsx


In [691]:
filename4 = next(iter(dist))

# Análise Ferramenta de Vendas

In [692]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [693]:
data = '/content/'+filename4
df_sale = pd.read_excel(data)
pd.options.display.max_columns=None
df_sale.nunique()

,0
Distributor (subsidiary),1
WACKER Business Team Initials,1
WACKER Market Segment,0
WACKER Application,0
WACKER Material No.,56
WACKER Material Name,60
Customer No. (at distributor),0
Customer Name,1087
Corporate Group (of customer),0
Country code ship-to (of customer),1


### Gera chave para Fabricante

In [694]:
#Dicionários
PALAVRAS_IRRELEVANTES = {
    'industria', 'comercio', 'cosmetico', 'cosmeticos', 'tecnologia', 'ltda', 'me', 'eireli', 'sa', 'cia', 'comercial', 'ind', 'e',
    'produtos', 'servicos', 'serviço', 'do', 'da', 'de', 'dos', 'das', 'the', 'group', 'grupo', 'laboratorio',
    'inc', 'corp', 'corporation', 'associados', 'associado', 'associacao', 'associação', 'holding', 'importadora',
    'exportadora', 'importacao', 'importação', 'exportacao', 'exportação', 'distribuidora', 'distribuidor', 'fabricacao',
    'fabricante', 'comerciante', 'comercio', 'comércio', 'comercial', 'empresa', 'sociedade', 'unipessoal',
    'aerosol', 'aerossol', 'technologies', 'prod', 'cosmetica', 'laboratorios',
    'atacado', 'quimica', 'industrial', 'pesquisas', 'cosmetic', 'beauty', 'higien', 'administradora',
    'fragrancias', 'aerossois', 'brasil', 'manufacturing', 'farmaceuticos', 'higiene', 'limpeza',
    'com', 'instituto',
}

In [695]:
# Função para limpar texto
def limpar(texto):
    if pd.isnull(texto):
        return []
    texto = unidecode(str(texto)).lower()
    texto = re.sub(r'\s+', ' ', texto)
    texto = re.sub(r'[.,/\\-]', ' ', texto)
    return [p for p in texto.split(' ') if p]

# Função para extrair chave representativa
def extrair_chave(palavras):
    if not palavras:
        return ''
    relevantes = [p for p in palavras if p not in PALAVRAS_IRRELEVANTES]
    if not relevantes:
        relevantes = palavras
    if len(relevantes) >= 3:
        return f"{relevantes[0]} {relevantes[1]} {relevantes[2]}"
    if len(relevantes) >= 2:
        return f"{relevantes[0]} {relevantes[1]}"
    else:
        chave = relevantes[0]
    chave = chave.strip()
    chave = re.sub(r'\b(e|and|&|\+)\b$', ' ', chave)
    chave = re.sub(r'\s+', ' ', chave)
    return chave

# Função para gerar chaves para DataFrame com fallback para valor bruto da coluna1
def gerar_chaves(df, coluna1, coluna2=None, coluna3=None):
    palavras1 = df[coluna1].apply(limpar)
    palavras2 = df[coluna2].apply(limpar) if coluna2 else pd.Series([[]] * len(df), index=df.index)
    palavras3 = df[coluna3].apply(limpar) if coluna3 else pd.Series([[]] * len(df), index=df.index)
    def chave_final(idx):
        for lista in [palavras1.iloc[idx], palavras2.iloc[idx], palavras3.iloc[idx]]:
            chave = extrair_chave(lista)
            if chave:
                return chave
        return str(df[coluna1].iloc[idx]).strip().lower()  # fallback para valor bruto da coluna1

    return pd.Series([chave_final(i) for i in range(len(df))], index=df.index)

### Gera a Chave de Material

In [696]:
def formatar_material(coluna):
    def extrair_codigo(texto):
        #Tenta extrair o padrão: 2+ letras + 1+ números
        match = re.search(r'\b([A-Za-z]{2,}\s*\d{1,})\b', texto)
        if match:
            return match.group(1).upper().strip()
    return coluna.apply(extrair_codigo)

### Relatório de Vendas

In [697]:
def resumo_por_ano(df, ano):
    df_ano = df[df['Year of Invoice'] == str(ano)]
    return (
        df_ano.groupby(['KeyManuf','Region (State/Province) ship-to (of customer)'])
        .agg(
            **{f'Produtos_{ano}': ('WACKER Material Name', lambda x: len(set(i for i in x if i))),
               f'Volume_{ano}': ('Quantity actual period', 'sum'),
               f'Lista_{ano}': ('WACKER Material Name', lambda x: sorted(set(i for i in x if i)))}
        )
    )

In [698]:
# Filtra apenas linhas que contenham 'BELSIL' na descrição (case-insensitive)
df_sale = df_sale[df_sale['WACKER Material Name'].str.contains('BELSIL', case=False, na=False)]

# Padroniza fabricantes e distribuidores
df_sale['KeyManuf'] = gerar_chaves(df_sale, 'Customer Name')

# Limpa e padroniza a coluna 'Material'
df_sale['WACKER Material Name'] = formatar_material(df_sale['WACKER Material Name'].astype(str))

# garante que o ano seja string
df_sale['Year of Invoice'] = df_sale['Year of Invoice'].astype(str)

resumo_2024 = resumo_por_ano(df_sale, '2024')
resumo_2025 = resumo_por_ano(df_sale, '2025')

# Junta os resultados
df_final = pd.concat([resumo_2024, resumo_2025], axis=1).reset_index()

# Funções para itens adicionados e perdidos
def itens_adicionados(row):
    l2024 = set(row['Lista_2024']) if isinstance(row['Lista_2024'], list) else set()
    l2025 = set(row['Lista_2025']) if isinstance(row['Lista_2025'], list) else set()
    return sorted(l2025 - l2024)

def itens_perdidos(row):
    l2024 = set(row['Lista_2024']) if isinstance(row['Lista_2024'], list) else set()
    l2025 = set(row['Lista_2025']) if isinstance(row['Lista_2025'], list) else set()
    return sorted(l2024 - l2025)

# Função para unir todos os códigos vendidos (2023 e 2024)
def Itens_Vendidos(row):
    l2 = set(row['Lista_2024']) if isinstance(row['Lista_2024'], list) else set()
    l3 = set(row['Lista_2025']) if isinstance(row['Lista_2025'], list) else set()
    return sorted(l2 | l3)

# Cria as colunas de interesse
df_final['Itens_Adicionados'] = df_final.apply(itens_adicionados, axis=1)
df_final['Itens_Perdidos'] = df_final.apply(itens_perdidos, axis=1)
df_final['Itens_Vendidos_2024_2025'] = df_final.apply(Itens_Vendidos, axis=1)

df_final['Estado'] = df_final['Region (State/Province) ship-to (of customer)']
# Seleciona as colunas finais
df_final = df_final[
    ['KeyManuf', 'Estado',
     'Produtos_2024', 'Produtos_2025',
     'Volume_2024', 'Volume_2025',
     'Itens_Adicionados', 'Itens_Perdidos',
     'Itens_Vendidos_2024_2025']
]

# Salva o resultado em Excel
df_final.to_excel('Vendas_BELSIL_por_ano.xlsx', index=False)
df_final.head()

,KeyManuf,Estado,Produtos_2024,Produtos_2025,Volume_2024,Volume_2025,Itens_Adicionados,Itens_Perdidos,Itens_Vendidos_2024_2025
0,&co,SP,3.0,3.0,1002.0,419.5,[EG 6000],[EG 5],"[DM 5, EG 5, EG 6000, TMS 803]"
1,2k,SC,1.0,NaN,15.0,NaN,[],[GB 1020],[GB 1020]
2,3fa,GO,1.0,3.0,30.0,581.0,"[EG 5, GB 1020, OW 2100]",[DM 0],"[DM 0, EG 5, GB 1020, OW 2100]"
3,a & d,SC,3.0,5.0,1003.0,2067.0,"[ADM 8301, DM 6010, OW 2100, TMS 803]","[EG 5, PF 22]","[ADM 8301, ADM 9000, DM 6010, EG 5, OW 2100, P..."
4,a&s,SP,5.0,4.0,8958.0,5274.0,[],[DM 60000],"[DM 0, DM 60000, DM 6010, OW 2100, TMS 803]"


# Análise Relatórios Lançamentos

## Descritivo Lançamentos





In [699]:
#Carrega o banco de dados como tabela
data = '/content/'+filename
df_launches = pd.read_excel(data)
pd.options.display.max_columns = None
df_launches.nunique()

,0
Número do Produto,4757
Data de Publicação,246
Produto,3123
Marca,2819
Empresa,899
Categoria,3
Sub-Categoria,33
Descrição do Produto,4752
Preço por 100g/ml,3196
Preço em moeda local,1574


In [700]:
#Edita Coluna Ano
df_launches['Ano'] = pd.to_datetime(df_launches['Data de Publicação']).dt.year

##Upload INCI

In [701]:
data = '/content/'+filename2
df_inci = pd.read_excel(data)
pd.options.display.max_columns=None
df_inci.nunique()

,0
Produto,51
Ingrediente,50
Prioridade,3


## Função Analisa Ingredientes

In [702]:
#this checks if any combination of INCI as present in Ingredient
def verifica_ingrediente(formula,produto,material):
  quantidade = len(produto.difference(formula))
  if quantidade == 0:
    return material
  return None

In [703]:
#If last code is true, this returns the Descrição name
def procura_produtos(formula, produtos, materiais):
  formula = formula.copy()
  resultados = []
  for prod, mat in zip(produtos, materiais):
    resultado = verifica_ingrediente(formula, prod, mat)
    if resultado is not None:
      formula = formula.difference(prod) ## Para remover os ingredientes já encontrados numa nova busca.
      resultados.append(resultado)
  return resultados

In [704]:
# Função para sinalizar ingredientes do dictOTHERS
def sinaliza_ingredientes(x):
    ingredientes = set().union(*x)  # une todos os sets/listas de ingredientes do grupo
    encontrados = set()
    for ing in ingredientes:
        for palavra in dictOTHERS:
            if palavra.lower() in ing.lower():
                encontrados.add(ing)
    return ', '.join(sorted(encontrados))

## Função Transpoe coluna

In [705]:
#this code transpose data. Used when we bring each category and the number of lauches.
def transpor (df, coluna, linha):
  for cat in df[coluna].unique():
    f = df[coluna] == cat
    df[cat] = df[f][linha]
    f = df[cat].isna()
    df.loc[f, cat] = df.loc[f, cat].apply(lambda x:[])

In [706]:
def to_set(x):
    s = set()
    for item in x:
        if isinstance(item, list):
            s.update(item)
        elif isinstance(item, str):
            s.add(item)
    return sorted(s)

In [707]:
def transpor_2 (df, coluna, linha):
  for cat in df[coluna].unique():
    f = df[coluna] == cat
    df[cat] = df[f][linha]
    f = df[cat].isna()
    df.loc[f, cat] = df.loc[f, cat].apply(lambda x:x)

## Contém Silicone?

In [708]:
#Dicionário Silicones geral
dictOTHERS = {'methicone':'1', 'Dimethicone':'1','methicone Crosspolymer':'1','methiconol':'1',
              'ylsiloxysilicate':'1', 'ylsilsesquioxane':'1', 'Disiloxane':'1', 'Silica':'1','siloxane':'1'}

In [709]:
df_launches['Silicone'] = df_launches['Ingredients (Standard form)'].str.extract('('+'|'.join(dictOTHERS)+')',expand=False).map(dictOTHERS)

## Gera chave de Material


In [710]:
#Edita tabela INCI

df_inci.dropna(inplace=True)
df_inci['Produto'] = formatar_material(df_inci['Produto'])

#Cria uma lista iterável dos ingredientes nos Produtos
df_inci['Ing'] = df_inci['Ingrediente'].str.split(', ').apply(set)
df_inci.sort_values('Prioridade', ascending=True, inplace=True)

In [711]:
#Dicionário de palavras a remover da coluna Ingredientes no Mintel
dictIng = {
    r'\s*and/or\s*': ',',
    r',\s*': ',',
    r'\s*,': ',',
    r'\s*\(and\)\s*': ','
}

In [712]:
# Cria coluna com produtos identificados
df_launches['Ingredients (Standard form)'] = df_launches['Ingredients (Standard form)'].astype(str)
#Cria uma lista iterável das ingredientes cosmeticos
df_launches['Ing'] = (
    df_launches['Ingredients (Standard form)']
    .replace(dictIng, regex=True)
    .str.split(',')
    .apply(set)
)

# Relaciona os ingredientes cosméticos
df_launches['Lançamentos'] = df_launches['Ing'].apply(
    lambda formulacao: procura_produtos(formulacao, df_inci['Ing'], df_inci['Produto'])
)

## Relatório de Lançamentos

In [713]:
df_launches['Fabricante']= df_launches['Fabricante'].astype(str)
df_launches['Marca']= df_launches['Marca'].astype(str)
df_launches['Empresa']= df_launches['Empresa'].astype(str)
df_launches['KeyManuf'] = gerar_chaves(df_launches, 'Fabricante', 'Empresa', 'Marca')
df_launches['KeyMarca'] = gerar_chaves(df_launches, 'Marca')
df_launches['KeyEmpresa'] = gerar_chaves(df_launches, 'Empresa')

In [714]:
# Dicionário de siglas e nomes de estados
estados = {
    'AC': 'Acre', 'AL': 'Alagoas', 'AP': 'Amapá', 'AM': 'Amazonas', 'BA': 'Bahia', 'CE': 'Ceará',
    'DF': 'Distrito Federal', 'ES': 'Espírito Santo', 'GO': 'Goiás', 'MA': 'Maranhão', 'MT': 'Mato Grosso',
    'MS': 'Mato Grosso do Sul', 'MG': 'Minas Gerais', 'PA': 'Pará', 'PB': 'Paraíba', 'PR': 'Paraná',
    'PE': 'Pernambuco', 'PI': 'Piauí', 'RJ': 'Rio de Janeiro', 'RN': 'Rio Grande do Norte',
    'RS': 'Rio Grande do Sul', 'RO': 'Rondônia', 'RR': 'Roraima', 'SC': 'Santa Catarina',
    'SP': 'São Paulo', 'SE': 'Sergipe', 'TO': 'Tocantins'
}

def extrair_estado(texto):
    if pd.isnull(texto):
        return None
    texto = str(texto).strip().lower()
    # Procura por sigla
    for sigla in estados:
        if re.search(r'\b' + re.escape(sigla.lower()) + r'\b', texto):
            return sigla
    # Procura por nome do estado
    for nome, sigla in estados_nome_para_sigla.items():
        if nome in texto:
            return sigla
    return None
# Inverte para buscar por nome também
estados_nome_para_sigla = {v.lower(): k for k, v in estados.items()}


# Exemplo: supondo que sua coluna de empresa é 'Fabricante'
df_launches['Estado'] = df_launches['Manufacturer Company Address'].apply(extrair_estado)

# Preencher os valores ausentes de 'Estado' com base em 'KeyManuf'
missing_estado = df_launches[df_launches['Estado'].isna()]
for index, row in missing_estado.iterrows():
    keymanuf = row['KeyManuf']

    # Buscar registros com o mesmo 'KeyManuf' e 'Estado' não nulo
    estado_valido = df_launches[
        (df_launches['KeyManuf'] == keymanuf) &
        (df_launches['Estado'].notna())
    ]['Estado'].unique()

    # Se houver apenas um valor único de 'Estado', preencher
    if len(estado_valido) == 1:
        df_launches.at[index, 'Estado'] = estado_valido[0]

In [715]:
enderecos_por_empresa = df_launches.dropna(subset=['Local de fabricação']).groupby('KeyManuf')['Local de fabricação'].first().to_dict()

# Passo 2: Preencher os endereços faltantes com base no dicionário
df_launches['Endereco'] = df_launches.apply(lambda row: enderecos_por_empresa.get(row['KeyManuf'], row['Local de fabricação']), axis=1)


In [716]:
df_launches['indice']=1
transpor_2(df_launches, 'Categoria', 'indice')

In [717]:
#renomeia coluna categoria
df_launches.rename(columns={'Produtos para Pele':'Pele', 'Produtos para Cabelos':'Cabelos',
                            'Maquilagem': 'Make',
                            'Número do Produto':'Total Lançamentos'},inplace=True)

In [718]:
df_c = df_launches.groupby(['KeyManuf', 'Endereco', 'Estado'], dropna=False)[['Pele', 'Cabelos', 'Make']].apply(lambda x:x.count())

In [719]:
#cria tabela que lista o fabricante, o estado e o total de lançamentos, quais tem silicone,
df_b = df_launches.groupby(['KeyManuf','Endereco','Estado'], dropna=False).agg({
    'KeyMarca': to_set,
    'KeyEmpresa': to_set,
    'Total Lançamentos': 'count',
    'Silicone': 'count',
    'Lançamentos': to_set,
    'Ing': sinaliza_ingredientes
})

In [720]:
#Une tabelas anteriores
df_launches_fab = pd.concat([df_c, df_b], axis=1).reset_index()

In [721]:
df_launches_fab.to_excel('Relatório Lançamentos.xlsx')
df_launches_fab.head()

,KeyManuf,Endereco,Estado,Pele,Cabelos,Make,KeyMarca,KeyEmpresa,Total Lançamentos,Silicone,Lançamentos,Ing
0,& business,Itália,NaN,0,4,0,"[alfaparf professional semi, alfaparf style st...",[& business],4,2,[ADM 9000],"Amodimethicone, Potassium Dimethicone PEG-7 Pa..."
1,&co,Brasil,SP,6,1,3,"[beyoung, ollie, pink cheeks, pinkcheeks, pink...","[&co, ollie]",10,8,"[ES 3007, GB 150, TMS 803]","C30-45 Alkyl Methicone, Cyclomethicone, Cyclop..."
2,3fa,Brasil,GO,0,4,0,"[we pink liberte, we pink obsessed, weblue]",[3fa],4,4,[GB 1020],"Cyclopentasiloxane, Dimethicone, Dimethiconol"
3,5s 5s,Brasil,SP,1,0,0,[fenzza make up],[5s 5s],1,0,[],
4,a & a,NaN,MG,1,0,0,[oroskin],[a & a],1,1,[OW 2100],PEG-12 Dimethicone


## Agrupar relatório Vendas e Projetos

In [722]:
#agrupa lançamentos com vendas e projetos
df_launches_vend_proj = df_launches_fab.merge(df_final, on=['KeyManuf', 'Estado'], how='outer')

In [723]:
df_launches_vend_proj['Tem_Lancamento'] = ~df_launches_vend_proj['Total Lançamentos'].isna()
df_launches_vend_proj['Tem_Venda'] = (
    ~df_launches_vend_proj['Volume_2024'].isna() &
    ~df_launches_vend_proj['Volume_2025'].isna()
)

In [724]:
df_launches_vend_proj.head()

,KeyManuf,Endereco,Estado,Pele,Cabelos,Make,KeyMarca,KeyEmpresa,Total Lançamentos,Silicone,Lançamentos,Ing,Produtos_2024,Produtos_2025,Volume_2024,Volume_2025,Itens_Adicionados,Itens_Perdidos,Itens_Vendidos_2024_2025,Tem_Lancamento,Tem_Venda
0,& business,Itália,NaN,0.0,4.0,0.0,"[alfaparf professional semi, alfaparf style st...",[& business],4.0,2.0,[ADM 9000],"Amodimethicone, Potassium Dimethicone PEG-7 Pa...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False
1,&co,Brasil,SP,6.0,1.0,3.0,"[beyoung, ollie, pink cheeks, pinkcheeks, pink...","[&co, ollie]",10.0,8.0,"[ES 3007, GB 150, TMS 803]","C30-45 Alkyl Methicone, Cyclomethicone, Cyclop...",3.0,3.0,1002.0,419.5,[EG 6000],[EG 5],"[DM 5, EG 5, EG 6000, TMS 803]",True,True
2,2k,NaN,SC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,15.0,NaN,[],[GB 1020],[GB 1020],False,False
3,3fa,Brasil,GO,0.0,4.0,0.0,"[we pink liberte, we pink obsessed, weblue]",[3fa],4.0,4.0,[GB 1020],"Cyclopentasiloxane, Dimethicone, Dimethiconol",1.0,3.0,30.0,581.0,"[EG 5, GB 1020, OW 2100]",[DM 0],"[DM 0, EG 5, GB 1020, OW 2100]",True,True
4,5s 5s,Brasil,SP,1.0,0.0,0.0,[fenzza make up],[5s 5s],1.0,0.0,[],,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False


# Generate a Report will all information

In [725]:
df_launches_vend_proj.to_excel('Final.xlsx')